In [165]:
from FormUtils import pyForm, capture_physics_expr

In [166]:
%%pyForm Running

* Process: Running

#-
* Above suppresses extra output
Off Statistics;
Off FinalStats;
#include SquareAmplitude.h

Symbols e, Mmuon, Melec;
Vectors k, kf1, kf2;
Symbols denom1, denom2, fx;


Local MLO = (e^2) * (UB(i1, p3, Melec ) * g(i1, i2, mu1) * U(i2, p1, Melec))
            * phprop(mu1, mu2, q)
            * (UB(i3, p4, Mmuon) * g(i3, i4, mu2) * U(i4, p2, Mmuon));

Local MNLO = -1* (e^4) 
             * (UB(i1, p3, Melec) * g(i1, i2, mu1) * U(i2, p1, Melec)) 
             * phprop(mu1, mu3, q) 
             * g(i11, i12, mu3) * fprop(i12, i13, kf1, Melec) 
             * g(i13, i14, mu4) * fprop(i14, i11, kf2, Melec)
             * phprop(mu4, mu2, q)
             * (UB(i7, p4, Mmuon) * g(i7, i8, mu2) * U(i8, p2, Mmuon));

Local MTotal = MLO+MNLO;
#call squareamplitude(MLO, MsqLO)
#call squareamplitude(MNLO, MsqNLO)
#call squareamplitude(MTotal, MsqTotal)
.sort
Multiply 1/4;
.sort
Drop drop MsqNLO, MsqTotal, MLO, MNLO, MTotal ;
Local MInt = MsqTotal - MsqLO - MsqNLO;
.sort


* --- KINEMATIC DEFINITIONS: VACUUM POLARIZATION ---
* q  = p1 - p3           : Momentum transfer between electron and muon lines
* t  = q.q               : Mandelstam variable t (photon momentum squared)
* k  = loop momentum     : Integration variable for the fermion loop
* kf1 = k                : Momentum of the first fermion in the bubble
* kf2 = k - q            : Momentum of the second fermion in the bubble (cons. of momentum)



* --- MASSLESS APPROXIMATION ---
id Melec = 0;
id Mmuon = 0;
#call Mandelstam2To2(p1,p2,p3,p4,0,0,0,0)
.sort
Format C;
#write <RunningBeforeSubst.txt> "%e;", MInt;
Format;
Print+s MInt;
.sort

* Replace the propagator function
id prop(q.q) = 1/t;
id prop(-Melec^2 + kf1.kf1) = 1/denom1;
id prop(-Melec^2 + kf2.kf2) = 1/denom2;
* Replace dot products involving loop momentum k
* Apply shift 
id kf1.p1? = fx*(t/2);;
id kf2.p2? = (fx-1)*(-t/2);
id kf1.kf2 = fx*(fx-1)*t;
.sort

* Print 
Format;
Print+s MInt;
Print+s MsqLO;
* Save to file 
Format C;
#write <RunningLO.txt> "%e;", MsqLO;
#write <Running.txt> "%e;", MInt;
.sort

.end

FORM 5.0.0 (Jan 27 2026, v5.0.0)                 Run: Fri Apr 17 18:12:40 2026
    
    * Process: Running
    
    #-

   MInt =
       + 16*prop(q.q)^3*prop( - Melec^2 + kf1.kf1)*prop( - Melec^2 + kf2.kf2)*
      p1.kf1*p2.kf2*s*e^6
       + 32*prop(q.q)^3*prop( - Melec^2 + kf1.kf1)*prop( - Melec^2 + kf2.kf2)*
      p1.kf1*p3.kf2*t*e^6
       - 16*prop(q.q)^3*prop( - Melec^2 + kf1.kf1)*prop( - Melec^2 + kf2.kf2)*
      p1.kf1*p4.kf2*u*e^6
       + 16*prop(q.q)^3*prop( - Melec^2 + kf1.kf1)*prop( - Melec^2 + kf2.kf2)*
      p1.kf2*p2.kf1*s*e^6
       + 32*prop(q.q)^3*prop( - Melec^2 + kf1.kf1)*prop( - Melec^2 + kf2.kf2)*
      p1.kf2*p3.kf1*t*e^6
       - 16*prop(q.q)^3*prop( - Melec^2 + kf1.kf1)*prop( - Melec^2 + kf2.kf2)*
      p1.kf2*p4.kf1*u*e^6
       - 16*prop(q.q)^3*prop( - Melec^2 + kf1.kf1)*prop( - Melec^2 + kf2.kf2)*
      p2.kf1*p3.kf2*u*e^6
       + 32*prop(q.q)^3*prop( - Melec^2 + kf1.kf1)*prop( - Melec^2 + kf2.kf2)*
      p2.kf1*p4.kf2*t*e^6
       - 16*prop(q.q)^3*prop( 

In [167]:
import numpy as np
import sympy as sp



form_expr_LO = capture_physics_expr("scripts/RunningLO.txt")
form_expr_Int = capture_physics_expr("scripts/Running.txt")

t, s, u, e = sp.symbols("t s u e", real=True)
fx = sp.Symbol("fx", real=True) 
denom1, denom2 = sp.symbols("denom1 denom2")

mapping = {
    'fx': fx, 
    's': s, 
    't': t, 
    'u': u, 
    'e': e, 
    'denom1': denom1, 
    'denom2': denom2
}

MLO = sp.parse_expr(form_expr_LO,local_dict=mapping)
MInt= sp.parse_expr(form_expr_Int,local_dict=mapping)
print("MLO :")
sp.pprint(MLO)
print("\n")
print("MInt")
sp.pprint(MInt)
print("\n")

# We replace 1/denom with 1.
# We will have to put back the integral over it 
MInt_num = MInt.subs({denom1: 1, denom2: 1})
print("MInt_num")
sp.pprint(MInt_num)
print("\n")
MInt_clean = sp.cancel(MInt_num)
poly_fx = sp.Poly(MInt_clean, fx)
a1 = poly_fx.nth(1)
a2 = poly_fx.nth(2)
print("Coefficient a1 (Kinematics):")
sp.pprint(sp.simplify(a1))
print("\nCoefficient a2 (Kinematics):")
sp.pprint(sp.simplify(a2))

Melec = sp.Symbol("Melec", real=True, positive=True)
mu2= sp.Symbol("mu2", real=True) 
# 3. Master Integrals
# Delta is the denominator of the Feynman parametrized integral
Delta = Melec**2 - fx*(1 - fx)*t
# Master Integrals for 1/Delta^2
I1 = sp.integrate(fx / Delta**2, (fx, 0, 1))
I2 = sp.integrate(fx**2 / Delta**2, (fx, 0, 1))
# Logarithmic scaling part
mu2 = sp.Symbol("mu2", real=True)
log_scale = sp.log(-t / mu2)
L1 = log_scale * I1
L2 = log_scale * I2


# Combine with the Master Integrals
# This is the integrated interference term
M_int_final = a1 * L1 + a2 * L2

# High energy limit (multiply to avoid pole)
M_int_final_limit = sp.limit(M_int_final * Melec**2, Melec, 0)
# Calculate the Ratio 
running_part = M_int_final_limit / MLO
# The beta is the coefficient in front of  log(-t/mu2)
beta_raw = sp.simplify(running_part / log_scale)
beta_final = sp.simplify(beta_raw)
beta_final_simplified = sp.simplify(beta_final.subs(t, -s - u))
print("Final Simplified Beta Result:")
sp.pprint(beta_final_simplified)


MLO :
   4  2      4  2
2⋅e ⋅s    2⋅e ⋅u 
─────── + ───────
   2         2   
  t         t    


MInt
        6   2             6   2           6   2             6     2            ↪
    16⋅e ⋅fx ⋅s       32⋅e ⋅fx        16⋅e ⋅fx ⋅u        8⋅e ⋅fx⋅s         16⋅ ↪
- ─────────────── - ───────────── + ─────────────── - ──────────────── + ───── ↪
  denom₁⋅denom₂⋅t   denom₁⋅denom₂   denom₁⋅denom₂⋅t                  2   denom ↪
                                                      denom₁⋅denom₂⋅t          ↪

↪  6                 6               6                 6     2   
↪ e ⋅fx⋅s        40⋅e ⋅fx        16⋅e ⋅fx⋅u         8⋅e ⋅fx⋅u    
↪ ────────── + ───────────── - ─────────────── - ────────────────
↪ ₁⋅denom₂⋅t   denom₁⋅denom₂   denom₁⋅denom₂⋅t                  2
↪                                                denom₁⋅denom₂⋅t 


MInt_num
      6   2                     6   2        6     2       6                   ↪
  16⋅e ⋅fx ⋅s       6   2   16⋅e ⋅fx ⋅u   8⋅e ⋅fx⋅s    16⋅e ⋅fx⋅s    